In [0]:
%sql
SELECT * FROM ECOMMERCE.BRONZE.PAYMENTS LIMIT 10;

## Loading libraries for future usage

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


In [0]:
payments_df = (
    spark.read.table("ecommerce.bronze.payments")
    .withColumn("payment_method", col("payment_method").cast("STRING"))
    .withColumn("payment_amount", col("payment_amount").cast("decimal(18,2)"))
    .withColumn("payment_status", col("payment_status").cast("string"))
    .filter(col("payment_amount") > 0)
    .dropDuplicates()
)

## Standardize payment method and paymnt status column

In [0]:
payments_df = payments_df.withColumn(
    "payment_method", upper(trim(col("payment_method")))
)

In [0]:
payments_df = payments_df.withColumn(
    "payment_status", upper(trim(col("payment_status")))
)

## Idetify failed successful and refunds

In [0]:
payments_df = payments_df.withColumn('failed_payments',when(col('payment_status')=='FAILED',1).otherwise(0))
payments_df = payments_df.withColumn('successful_payments',when(col('payment_status')=='SUCCESS',1).otherwise(0))
payments_df = payments_df.withColumn('refund_payments',when(col('payment_status')=='REFUNDED',1).otherwise(0))


In [0]:
payments_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.payments"
)